# Hedge vs AdaHedge — reproducing the results

This notebook reproduces all the results of the report with one click:

1. it clones the repository and installs the requirements;
2. it runs `main.py` (all the experiments: 3 regimes × 100 repetitions × 6 algorithms, a few minutes);
3. it checks that the regenerated numbers are identical to the ones committed in `results/` and used in the report;
4. it shows the final-regret table and the figures.

The algorithms are not re-implemented here: the notebook only runs the code in `src/` through `main.py`.

## 1. Clone the repository and install the requirements

In [ ]:
import os

REPO_URL = "https://github.com/bruskez/Hedge-vs.-AdaHedge.git"
REPO_DIR = "Hedge-vs.-AdaHedge"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 2. Run all the experiments

This regenerates every file in `results/` (figures, CSV, JSON and LaTeX table).

In [ ]:
import subprocess, sys, time

start = time.time()
run = subprocess.run([sys.executable, "main.py"], capture_output=True, text=True)
# main.py prints progress with carriage returns: keep only the final lines
print("\n".join(l for l in run.stdout.replace("\r", "\n").splitlines()
                if l.strip() and "repetition" not in l))
print(run.stderr)
run.check_returncode()
print(f"Elapsed time: {time.time() - start:.0f} s")

## 3. Check that the results match the committed ones

The regenerated numerical results are compared with the versions stored in the repository (`git show HEAD:...`).

In [ ]:
import csv, io, json, subprocess

def committed(path):
    return subprocess.run(["git", "show", f"HEAD:{path}"], capture_output=True,
                          text=True, check=True).stdout

def rows(text):
    return [r for r in csv.reader(io.StringIO(text.replace("\r", "")))]

with open("results/final_regret.csv", newline="") as f:
    csv_ok = rows(f.read()) == rows(committed("results/final_regret.csv"))
with open("results/diagnostics.json") as f:
    json_ok = json.load(f) == json.loads(committed("results/diagnostics.json"))

print("final_regret.csv identical to the committed version:", csv_ok)
print("diagnostics.json identical to the committed version:", json_ok)

## 4. Final regret $R(T)$ (mean ± standard deviation over 100 repetitions)

In [ ]:
with open("results/final_regret.csv", newline="") as f:
    table = list(csv.DictReader(f))

regimes = ["stochastic", "adversarial", "low_gap"]
algorithms = list(dict.fromkeys(r["algorithm"] for r in table))
cell = {(r["regime"], r["algorithm"]): f'{float(r["mean_final_regret"]):8.1f} ± {float(r["std_final_regret"]):5.1f}'
        for r in table}

print(f'{"algorithm":<24}' + "".join(f"{reg:>18}" for reg in regimes))
for a in algorithms:
    print(f"{a:<24}" + "".join(f"{cell[(reg, a)]:>18}" for reg in regimes))

## 5. Figures

The figures are saved as PDF by `main.py`; they are rendered here as images for display.

In [ ]:
!pip install -q pymupdf
import pymupdf  # used only to display the PDF figures
from IPython.display import Image, display

for name in ["stochastic", "adversarial", "low_gap", "eta_sensitivity"]:
    page = pymupdf.open(f"results/{name}.pdf")[0]
    print(name)
    display(Image(data=page.get_pixmap(dpi=130).tobytes("png")))